In [1]:
"""
Bradford Air Quality Dashboard (Single-page)
- Left sidebar filters (site + date range)
- Main page: KPIs + trends + site rankings + exceedance summary + QA tables
- Reads processed CSVs from: data/processed/

How to run:
1) Ensure you have exported: data/processed/air_quality_hourly_clean.csv
   Optional (if available):
   - data/processed/air_quality_daily_by_site.csv
   - data/processed/qa_commissioning_by_site_pollutant.csv
   - data/processed/qa_negative_values_summary.csv

2) Install:
   pip install dash dash-bootstrap-components plotly pandas numpy

3) Run (recommended as a script):
   python dashboards/app.py

If running in a notebook, __file__ is not defined—this code handles that.
"""

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio  # noqa: F401 (kept for future HTML export if needed)

from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc


# =========================
# PATHS (script + notebook safe)
# =========================
try:
    # When running as dashboards/app.py
    BASE_DIR = Path(__file__).resolve().parents[1]
except NameError:
    # When running inside a notebook, set cwd to repo root OR dashboards and adjust
    # If you run from dashboards/ folder in notebook: BASE_DIR = Path.cwd().parent
    # If you run from repo root in notebook:      BASE_DIR = Path.cwd()
    cwd = Path.cwd()
    BASE_DIR = cwd.parent if cwd.name.lower() == "dashboards" else cwd

PRO_DATA = BASE_DIR / "data" / "processed"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

HOURLY_CSV = PRO_DATA / "air_quality_hourly_clean.csv"
DAILY_CSV = PRO_DATA / "air_quality_daily_by_site.csv"  # optional
QA_COMM_CSV = PRO_DATA / "qa_commissioning_by_site_pollutant.csv"  # optional
QA_NEG_CSV = PRO_DATA / "qa_negative_values_summary.csv"  # optional


# =========================
# LOAD DATA
# =========================
if not HOURLY_CSV.exists():
    raise FileNotFoundError(
        f"Missing file: {HOURLY_CSV}\n"
        "Export your cleaned hourly data first:\n"
        "df_clean.to_csv(data/processed/air_quality_hourly_clean.csv, index=False)"
    )

df = pd.read_csv(HOURLY_CSV, parse_dates=["datetime", "date"])
df["site"] = df["site"].astype(str)

# daily (load if present, otherwise compute from hourly)
if DAILY_CSV.exists():
    daily = pd.read_csv(DAILY_CSV, parse_dates=["date"])
else:
    daily = (
        df.groupby(["site", "date"], as_index=False)
          .agg(
              pm25_mean=("pm25", "mean"),
              pm10_mean=("pm10", "mean"),
              pm25_obs=("pm25", "count"),
              pm10_obs=("pm10", "count"),
          )
    )

commissioning = pd.read_csv(QA_COMM_CSV, parse_dates=["commissioned_from", "commissioned_to"]) if QA_COMM_CSV.exists() else pd.DataFrame()
negative_summary = pd.read_csv(QA_NEG_CSV) if QA_NEG_CSV.exists() else pd.DataFrame()


# =========================
# CONSTANTS (context lines)
# =========================
WHO_PM25_24H = 15.0  # µg/m³ (WHO 2021 guideline, 24-hour PM2.5)
WHO_PM10_24H = 45.0  # µg/m³ (WHO 2021 guideline, 24-hour PM10)


# =========================
# HELPERS
# =========================
def polish(fig, title: str, y_title: str = "µg/m³"):
    fig.update_layout(
        title=title,
        template="plotly_white",
        hovermode="x unified",
        legend_title_text="",
        margin=dict(l=40, r=20, t=60, b=40),
        height=420,
    )
    fig.update_xaxes(title_text="Date", showline=True, linewidth=1, linecolor="black", mirror=True)
    fig.update_yaxes(title_text=y_title, showline=True, linewidth=1, linecolor="black", mirror=True)
    return fig


def compute_site_ranking(daily_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    site_ranking = (
        daily_df.groupby("site", as_index=False)
                .agg(
                    pm25_avg_daily=("pm25_mean", "mean"),
                    pm10_avg_daily=("pm10_mean", "mean"),
                    days_observed=("date", "nunique"),
                )
    )
    site_ranking[["pm25_avg_daily", "pm10_avg_daily"]] = site_ranking[["pm25_avg_daily", "pm10_avg_daily"]].round(2)
    rank_pm25 = site_ranking.sort_values("pm25_avg_daily", ascending=False).reset_index(drop=True)
    rank_pm10 = site_ranking.sort_values("pm10_avg_daily", ascending=False).reset_index(drop=True)
    return rank_pm25, rank_pm10


def compute_exceedance(daily_df: pd.DataFrame,
                       pm25_thr: float = WHO_PM25_24H,
                       pm10_thr: float = WHO_PM10_24H) -> pd.DataFrame:
    tmp = daily_df.copy()
    tmp["pm25_exceed"] = tmp["pm25_mean"] > pm25_thr
    tmp["pm10_exceed"] = tmp["pm10_mean"] > pm10_thr

    out = (
        tmp.groupby("site", as_index=False)
           .agg(
               days_observed=("date", "nunique"),
               pm25_exceed_days=("pm25_exceed", "sum"),
               pm10_exceed_days=("pm10_exceed", "sum"),
           )
    )
    out["pm25_exceed_pct"] = (out["pm25_exceed_days"] / out["days_observed"] * 100).round(1)
    out["pm10_exceed_pct"] = (out["pm10_exceed_days"] / out["days_observed"] * 100).round(1)
    return out.sort_values("site").reset_index(drop=True)


def kpi_cards(filtered_daily: pd.DataFrame):
    pm25_avg = float(filtered_daily["pm25_mean"].mean()) if filtered_daily["pm25_mean"].notna().any() else np.nan
    pm10_avg = float(filtered_daily["pm10_mean"].mean()) if filtered_daily["pm10_mean"].notna().any() else np.nan
    n_sites = int(filtered_daily["site"].nunique())
    n_days = int(filtered_daily["date"].nunique())

    def card(title, value):
        return dbc.Card(
            dbc.CardBody([
                html.Div(title, className="text-muted"),
                html.H4(value, className="mb-0"),
            ]),
            className="shadow-sm",
        )

    return [
        dbc.Col(card("Sites selected", f"{n_sites}"), md=3),
        dbc.Col(card("Days in view", f"{n_days}"), md=3),
        dbc.Col(card("Avg PM2.5 (daily)", f"{pm25_avg:.2f}" if np.isfinite(pm25_avg) else "—"), md=3),
        dbc.Col(card("Avg PM10 (daily)", f"{pm10_avg:.2f}" if np.isfinite(pm10_avg) else "—"), md=3),
    ]


# =========================
# APP
# =========================
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app.title = "Bradford Air Quality Dashboard"

all_sites = sorted(daily["site"].unique().tolist())
min_date = daily["date"].min()
max_date = daily["date"].max()

sidebar = dbc.Card(
    dbc.CardBody([
        html.H5("Filters", className="mb-3"),

        html.Label("Sites"),
        dcc.Dropdown(
            id="site_dropdown",
            options=[{"label": s, "value": s} for s in all_sites],
            value=all_sites,
            multi=True,
            placeholder="Choose site(s)...",
        ),
        html.Div(className="mb-3"),

        html.Label("Date range"),
        dcc.DatePickerRange(
            id="date_range",
            start_date=min_date,
            end_date=max_date,
            min_date_allowed=min_date,
            max_date_allowed=max_date,
            display_format="DD/MM/YYYY",
        ),
        html.Hr(),

        html.Div(
            "Tip: Click legend items to hide/show lines. Hover to compare values on the same date.",
            className="small text-muted"
        ),
    ]),
    className="shadow-sm",
)

main = html.Div([
    html.H2("Bradford Air Quality – Winter Period Dashboard", className="mb-0"),
    html.Div("PM2.5 & PM10 daily summaries with QA context.", className="text-muted mb-3"),

    dbc.Row(id="kpi_row", className="g-3 mb-3"),

    dbc.Row([dbc.Col(dcc.Graph(id="fig_overall"), md=12)], className="mb-3"),

    dbc.Row([
        dbc.Col(dcc.Graph(id="fig_pm10"), md=6),
        dbc.Col(dcc.Graph(id="fig_pm25"), md=6),
    ], className="g-3 mb-3"),

    dbc.Row([
        dbc.Col(dcc.Graph(id="fig_rank_pm25"), md=6),
        dbc.Col(dcc.Graph(id="fig_rank_pm10"), md=6),
    ], className="g-3 mb-3"),

    dbc.Row([
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("Exceedance summary (contextual WHO 24h guidelines)", className="mb-2"),
            html.Div(id="exceed_table"),
        ]), className="shadow-sm"), md=12),
    ], className="mb-3"),

    dbc.Row([
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("QA – Commissioning & uptime (from processed QA output)", className="mb-2"),
            html.Div(id="qa_comm_table"),
        ]), className="shadow-sm"), md=12),
    ], className="mb-3"),

    dbc.Row([
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("QA – Negative values summary (from processed QA output)", className="mb-2"),
            html.Div(id="qa_neg_table"),
        ]), className="shadow-sm"), md=12),
    ]),
])

app.layout = dbc.Container(
    fluid=True,
    children=[
        dbc.Row([
            dbc.Col(sidebar, md=3, lg=3),
            dbc.Col(main, md=9, lg=9),
        ], className="mt-3")
    ]
)


# =========================
# CALLBACK
# =========================
@app.callback(
    Output("kpi_row", "children"),
    Output("fig_overall", "figure"),
    Output("fig_pm10", "figure"),
    Output("fig_pm25", "figure"),
    Output("fig_rank_pm25", "figure"),
    Output("fig_rank_pm10", "figure"),
    Output("exceed_table", "children"),
    Output("qa_comm_table", "children"),
    Output("qa_neg_table", "children"),
    Input("site_dropdown", "value"),
    Input("date_range", "start_date"),
    Input("date_range", "end_date"),
)
def update_dashboard(selected_sites, start_date, end_date):
    if not selected_sites:
        selected_sites = all_sites

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    f_daily = daily[
        (daily["site"].isin(selected_sites)) &
        (daily["date"] >= start_date) &
        (daily["date"] <= end_date)
    ].copy()

    # KPIs
    kpis = kpi_cards(f_daily)

    # Overall trend (avg across selected sites)
    overall = (
        f_daily.groupby("date", as_index=False)
              .agg(pm25_mean=("pm25_mean", "mean"),
                   pm10_mean=("pm10_mean", "mean"))
    )

    overall_long = overall.melt(
        id_vars="date",
        value_vars=["pm25_mean", "pm10_mean"],
        var_name="pollutant",
        value_name="value",
    )
    overall_long["pollutant"] = overall_long["pollutant"].map({"pm25_mean": "PM2.5", "pm10_mean": "PM10"})

    fig_overall = px.line(overall_long, x="date", y="value", color="pollutant")
    fig_overall = polish(fig_overall, "Overall daily trend (average across selected sites)")

    # By-site PM10
    fig_pm10 = px.line(f_daily, x="date", y="pm10_mean", color="site")
    fig_pm10 = polish(fig_pm10, "Daily PM10 by site", y_title="PM10 (µg/m³)")
    fig_pm10.add_hline(
        y=WHO_PM10_24H,
        line_dash="dash",
        line_color="grey",
        line_width=1,
        annotation_text="WHO PM10 24h (45 µg/m³)",
        annotation_position="top",
        annotation_font_size=11,
    )

    # By-site PM2.5
    fig_pm25 = px.line(f_daily, x="date", y="pm25_mean", color="site")
    fig_pm25 = polish(fig_pm25, "Daily PM2.5 by site", y_title="PM2.5 (µg/m³)")
    fig_pm25.add_hline(
        y=WHO_PM25_24H,
        line_dash="dash",
        line_color="grey",
        line_width=1,
        annotation_text="WHO PM2.5 24h (15 µg/m³)",
        annotation_position="top",
        annotation_font_size=11,
    )

    # Rankings
    rank_pm25, rank_pm10 = compute_site_ranking(f_daily)

    fig_rank_pm25 = px.bar(
        rank_pm25,
        x="pm25_avg_daily",
        y="site",
        orientation="h",
        hover_data={"days_observed": True},
    )
    fig_rank_pm25.update_layout(template="plotly_white", height=420, title="Site ranking by average daily PM2.5")
    fig_rank_pm25.update_yaxes(categoryorder="total ascending")
    fig_rank_pm25.update_xaxes(title_text="Average daily PM2.5 (µg/m³)")
    fig_rank_pm25.update_yaxes(title_text="Site")

    fig_rank_pm10 = px.bar(
        rank_pm10,
        x="pm10_avg_daily",
        y="site",
        orientation="h",
        hover_data={"days_observed": True},
    )
    fig_rank_pm10.update_layout(template="plotly_white", height=420, title="Site ranking by average daily PM10")
    fig_rank_pm10.update_yaxes(categoryorder="total ascending")
    fig_rank_pm10.update_xaxes(title_text="Average daily PM10 (µg/m³)")
    fig_rank_pm10.update_yaxes(title_text="Site")

    # Exceedance summary table
    exceed = compute_exceedance(f_daily, pm25_thr=WHO_PM25_24H, pm10_thr=WHO_PM10_24H)
    exceed_table = dbc.Table.from_dataframe(exceed, striped=True, bordered=True, hover=True, responsive=True)

    # QA tables (optional)
    if commissioning.empty:
        qa_comm = html.Div("QA commissioning file not found in data/processed.", className="text-muted")
    else:
        comm_f = commissioning[commissioning["site"].isin(selected_sites)].copy()
        qa_comm = dbc.Table.from_dataframe(comm_f, striped=True, bordered=True, hover=True, responsive=True)

    if negative_summary.empty:
        qa_neg = html.Div("QA negative values file not found in data/processed.", className="text-muted")
    else:
        neg_f = negative_summary[negative_summary["site"].isin(selected_sites)].copy()
        qa_neg = dbc.Table.from_dataframe(neg_f, striped=True, bordered=True, hover=True, responsive=True)

    return (
        kpis,
        fig_overall,
        fig_pm10,
        fig_pm25,
        fig_rank_pm25,
        fig_rank_pm10,
        exceed_table,
        qa_comm,
        qa_neg,
    )


if __name__ == "__main__":
    # Dash new API
    app.run(debug=True, host="127.0.0.1", port=8050)